# Гликемическая нагрузка (ГН) — сквозное демо

**ГН — главная метрика качества углеводов при диабете.** В отличие от
гликемического индекса (ГИ), ГН **аддитивна**: её можно складывать по
ингредиентам блюда и за день.

```
ГН порции = (ГИ / 100) × углеводы_порции(г)
         = (ГИ / 100) × carbs_per_100g × grams / 100
```

Поэтому на вопрос *«можно ли рис диабетику?»* ответ даёт именно ГН:
- **большая** порция белого риса (150 г) → ГН ≈ 30 (высокая, скачок сахара);
- **маленькая** порция (50 г) → ГН ≈ 10 (умеренная, допустимо);
- **мясо** (углеводов нет) → ГН = 0, на сахар не влияет.

В этом ноутбуке: инициализация БД → пользователь-диабетик → демонстрация
«рис + мясо» → сборка дня → проверки норм ГН → история.

> ⚠️ Данные о нутриентах берутся из Open Food Facts (41% заполненности) —
> некоторые продукты без КБЖУ. Поиск предпочитает продукты с заполненными
> данными.

## 1. Импорты и инициализация БД

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from diet import db
from diet import UserProfile, calculate
from diet.gi import glycemic_load, gl_label
from diet.checks import check_gl_portion, check_gl_meal, check_gl_day, gl_targets_for

pd.set_option('display.unicode.east_asian_width', True)
pd.set_option('display.width', 120)

# Инициализируем схему БД (идемпотентно).
db.init_db()

# Проверяем, что продукты загружены.
with db.connect() as conn:
    n_products = conn.execute("SELECT COUNT(*) FROM products").fetchone()[0]
print(f"Продуктов в БД: {n_products:,}")
if n_products == 0:
    print('⚠️ БД пуста. Запустите: ./.venv/Scripts/python.exe scripts/06_load_db.py')

Продуктов в БД: 30,965


## 2. Пользователь-диабетик

Считаем норму КБЖУ для мужчины с диабетом 2 типа и сохраняем профиль в БД.

In [2]:
profile = UserProfile(
    sex='male', age=55, weight=90, height=175,
    activity='light', goal='maintain',
)
result = calculate(profile, formula='who', condition='diabetes_t2')

print(f"Цель: {result.target_kcal:.0f} ккал/день")
print(f"БЖУ: {result.protein_g:.0f}/{result.fat_g:.0f}/{result.carbs_g:.0f} г")
print(f"Клетчатка: {result.fiber_g:.0f} г")
print(f"Нормы ГН для диабета: {gl_targets_for('diabetes_t2')}  (приём ≤ meal, день < day)")

# Сохраняем в БД (чистим старые тестовые записи)
with db.transaction() as conn:
    conn.execute('DELETE FROM diary_entries')
    conn.execute('DELETE FROM users')
    uid = db.add_user(conn, profile, result, name='Демо-диабетик')
print(f"\nПользователь id={uid} сохранён")

Цель: 2676 ккал/день
БЖУ: 120/89/348 г
Клетчатка: 40 г
Нормы ГН для диабета: {'meal': 15, 'day': 80}  (приём ≤ meal, день < day)

Пользователь id=3 сохранён


## 3. Демонстрация «рис + мясо»

Главный тезис: **мясо не снижает ГН блюда — оно просто добавляет 0**.
Вся ГН приходит от углеводного компонента (рис).

In [3]:
# Находим рис и мясо в БД (поиск отдаёт продукты с заполненным КБЖУ)
with db.connect() as conn:
    rice = db.search_products(conn, 'рис длиннозерн', limit=1).iloc[0]
    meat = db.search_products(conn, 'говядин', limit=1).iloc[0]

print(f"Рис: {rice['name']}  | ГИ={rice['gi']} углев={rice['carbs_g']} г/100г")
print(f"Мясо: {meat['name']}  | ГИ={meat['gi']} углев={meat['carbs_g']} г/100г")

Рис: Рис длиннозерный  | ГИ=75 углев=78.0 г/100г
Мясо: Говядина  | ГИ=0 углев=2.5 г/100г


In [4]:
# ГН каждого компонента и блюда целиком.
# ВАЖНО: углеводы в БД — для СУХОГО риса (~78 г/100г), ГИ — для варёного.
# Поэтому берём небольшую массу «сухого эквивалента»
# (60 г сухого ≈ 180 г варёного риса).
grams_rice, grams_meat = 60, 200

gl_rice = glycemic_load(rice['gi'], rice['carbs_g'], grams_rice)
gl_meat = glycemic_load(meat['gi'], meat['carbs_g'], grams_meat)

print(f"Рис {grams_rice} г:  ГН = {gl_rice}  ({gl_label(gl_rice)[1]})")
print(f"Мясо {grams_meat} г: ГН = {gl_meat}  ({gl_label(gl_meat)[1]})")
print(f"Блюдо целиком:       ГН = {gl_rice + gl_meat}  ← мясо добавило 0")

Рис 60 г:  ГН = 35.1  (ГН 35.1 ≥20 — высокая)
Мясо 200 г: ГН = 0.0  (ГН 0 ≤10 — низкая)
Блюдо целиком:       ГН = 35.1  ← мясо добавило 0


### Проверка порции риса против нормы диабета

In [5]:
# check_gl_portion сравнивает ГН порции с нормой за приём (meal=15)
for g in [30, 60, 100, 150]:
    checks = check_gl_portion(rice['gi'], rice['carbs_g'], g, f'Рис {g} г', 'diabetes_t2')
    gl = glycemic_load(rice['gi'], rice['carbs_g'], g)
    status = checks[0].level if checks else 'ok'
    print(f'  Рис {g:>3} г → ГН {gl:>5}  [{status}]')

print('\nВывод: «рис нельзя» = «нельзя БОЛЬШУЮ порцию», а не «нельзя рис».')

  Рис  30 г → ГН  17.5  [warn]
  Рис  60 г → ГН  35.1  [danger]
  Рис 100 г → ГН  58.5  [danger]
  Рис 150 г → ГН  87.8  [danger]

Вывод: «рис нельзя» = «нельзя БОЛЬШУЮ порцию», а не «нельзя рис».


## 4. Сборка дня и итоги с ГН

Соберём день из нескольких приёмов и посчитаем итоги. Записи пишем в БД,
итоги агрегируются запросом.

In [6]:
today = db._today_utc()

with db.transaction() as conn:
    # Завтрак: гречка + мясо (гречка — цельное зерно, ГИ ниже)
    buckwheat = db.search_products(conn, 'гречнев', limit=1).iloc[0]
    db.add_entry(conn, uid, buckwheat['id'], 50, meal='breakfast', day=today)
    db.add_entry(conn, uid, meat['id'], 150, meal='breakfast', day=today)

    # Обед: рис + мясо + овощи(яблоко)
    apple = db.search_products(conn, 'яблок', limit=1).iloc[0]
    db.add_entry(conn, uid, rice['id'], 60, meal='lunch', day=today)
    db.add_entry(conn, uid, meat['id'], 200, meal='lunch', day=today)
    db.add_entry(conn, uid, apple['id'], 150, meal='lunch', day=today)

    # Ужин: мясо + овощи (минимум углеводов)
    db.add_entry(conn, uid, meat['id'], 200, meal='dinner', day=today)
    db.add_entry(conn, uid, apple['id'], 100, meal='dinner', day=today)

print(f'День {today}: записи добавлены')

День 2026-08-10: записи добавлены


In [7]:
# Итоги дня и по приёмам
with db.connect() as conn:
    day = db.day_totals(conn, uid, today)
    meals = {}
    for m, label in [('breakfast','Завтрак'), ('lunch','Обед'), ('dinner','Ужин')]:
        meals[label] = db.meal_totals(conn, uid, today, m)

print('=== ИТОГИ ЗА ДЕНЬ ===')
print(f"  ккал={day['kcal']}  Б={day['protein_g']}  Ж={day['fat_g']}  У={day['carbs_g']}")
print(f"  клетчатка={day['fiber_g']}  ГН={day['gl']}")
print()
rows = []
for label, t in meals.items():
    rows.append({'Приём': label, 'ккал': t['kcal'], 'Углеводы': t['carbs_g'], 'ГН': t['gl']})
pd.DataFrame(rows)

=== ИТОГИ ЗА ДЕНЬ ===
  ккал=1222.6  Б=65.8  Ж=46.4  У=133.8
  клетчатка=7.0  ГН=70.0



,Приём,ккал,Углеводы,ГН
0,Завтрак,376.3,42.0,18.4
1,Обед,544.2,72.8,45.0
2,Ужин,302.0,19.0,6.6


### Проверки норм ГН

In [8]:
print('Проверка ГН за день (норма 80):')
day_checks = check_gl_day(day['gl'], 'diabetes_t2')
if day_checks:
    for c in day_checks:
        print(f'  {c}')
else:
    print('  ✓ в пределах нормы')

print('\nПроверка ГН по приёмам (норма 15):')
for label in ['Завтрак', 'Обед', 'Ужин']:
    checks = check_gl_meal(meals[label]['gl'], 'diabetes_t2', label)
    if checks:
        print(f'  {checks[0]}')
    else:
        print(f"  {label}: ГН={meals[label]['gl']} ✓ в норме")

Проверка ГН за день (норма 80):
  ✓ в пределах нормы

Проверка ГН по приёмам (норма 15):
  ⚠ ГН за приём пищи (Завтрак) = 18 — превышена норма 15 на 3 при сахарный диабет 2 типа
  ⛔ ГН за приём пищи (Обед) = 45 — превышена норма 15 на 30 при сахарный диабет 2 типа
  Ужин: ГН=6.6 ✓ в норме


## 5. История по дням

Итоги за последние дни (КБЖУ + ГН) — для трекинга.

In [9]:
with db.connect() as conn:
    hist = db.day_history(conn, uid, days=7)
hist

,day,kcal,protein_g,fat_g,carbs_g,fiber_g,gl
0,2026-08-04,0.0,0.0,0.0,0.0,0.0,0.0
1,2026-08-05,0.0,0.0,0.0,0.0,0.0,0.0
2,2026-08-06,0.0,0.0,0.0,0.0,0.0,0.0
3,2026-08-07,0.0,0.0,0.0,0.0,0.0,0.0
4,2026-08-08,0.0,0.0,0.0,0.0,0.0,0.0
5,2026-08-09,0.0,0.0,0.0,0.0,0.0,0.0
6,2026-08-10,1222.6,65.8,46.4,133.8,7.0,70.0


## Итоги

| Что | Как работает |
|---|---|
| **ГН порции** | `gi/100 × углеводы × масса/100` — `diet.gi.glycemic_load` |
| **ГН блюда** | сумма ГН по ингредиентам (аддитивность) |
| **ГН приёма/дня** | агрегируется в `diet.db.meal_totals` / `day_totals` |
| **Нормы диабета** | приём ≤ 15, день < 80 — `conditions.gl_targets` |
| **Проверки** | `diet.checks.check_gl_portion/meal/day` |
| **Хранение** | SQLite (локально) → PostgreSQL (прод, `DIET_DB_URL`) |

**Ключевая идея:** ГИ продукта — это характеристика, а ГН — измеримая
метрика. Мясо даёт ГН=0, рис — основную ГН. «Рис нельзя диабетику» =
«нельзя большую порцию», а не абсолютный запрет.